In [26]:
import requests
import re
import json
from Bio import SeqIO
import subprocess
import sys

In [27]:
class MyFastaParser:
    def __init__(self, file_name):
        self.filename = file_name

    def _get_uniprot(self, accession):
        '''
        define http_function to get the data from Uniprot API
        '''
        url = f'https://rest.uniprot.org/uniprotkb/{accession}'
        return requests.get(url, headers={'Accept': 'application/json'})

    def _get_ensembl(self, id):
        '''
        define http_function to get the data from Ensembl API
        '''
        url = f'https://rest.ensembl.org/lookup/id/{id}?content-type=application/json'
        return requests.get(url)

    def _uniprot_parse_response(self, resp: dict):
        '''
        parse response from Uniprot and output
        organism, geneInfo, sequenceInfo, type

        do not forget to include error handling (e.g. for key errors)
        '''
        try:
            data = resp.json() if hasattr(resp, 'json') else resp
            
            accession = data.get('primaryAccession', 'Unknown')
            organism_data = data.get('organism', {})
            organism = organism_data.get('scientificName', organism_data) if isinstance(organism_data, dict) else organism_data
            
            output = {
                accession: {
                    'organism': organism,
                    'geneInfo': data.get('genes', []),
                    'sequenceInfo': data.get('sequence', {}),
                    'type': 'protein'
                }
            }
            return output
        except Exception as e:
            return {'error': str(e)}

    def _ensembl_parse_response(self, resp: dict):
        '''
        parse Ensembl response and output
        object_type, assembly_name, species, db_type, biotype, display_name, id, description, canonical_transcript, source
        '''
        try:
            data = resp.json() if hasattr(resp, 'json') else resp
                
            ensembl_id = data.get('id', 'Unknown')
            keys_to_extract = [
                'object_type', 'assembly_name', 'species', 'db_type', 'biotype', 
                'display_name', 'id', 'description', 'canonical_transcript', 'source'
            ]
            output = {
                ensembl_id: {k: data.get(k) for k in keys_to_extract if k in data}
            }
            return output
        except Exception as e:
            return {'error': str(e)}

    def _access_database(self, id, database, seq_description, seq_sequence) -> dict:
        # Calls 'get' and then 'parse' functions from HW2
        db_info = {}
        if database.lower() == 'uniprot':
            resp = self._get_uniprot(id)
            if resp.status_code == 200:
                db_info = self._uniprot_parse_response(resp)
            else:
                db_info = {'error': f'API returned status {resp.status_code}'}
        elif database.lower() == 'ensembl':
            resp = self._get_ensembl(id)
            if resp.status_code == 200:
                db_info = self._ensembl_parse_response(resp)
            else:
                db_info = {'error': f'API returned status {resp.status_code}'}
        else:
            db_info = {'error': 'Unknown database specified'}

        adb_output = {
            'description': seq_description,
            'sequence': str(seq_sequence),
            'database_name': database,
            'database_info': db_info
        }
        return adb_output

    def seqkit_stats(self) -> dict:
        try:
            result = subprocess.run(
                ['seqkit', 'stats', '-a', '-T', self.filename], 
                capture_output=True, 
                text=True, 
                check=True
            )
            lines = result.stdout.strip().split('\n')
            
            if len(lines) > 1:
                headers = lines[0].split('\t')
                values = lines[1].split('\t')
                
                stat_info = dict(zip(headers, values))
                
                seq_type = stat_info.get('type', 'Unknown')
                num_seqs = int(stat_info.get('num_seqs', 0))
                
                return {
                    'fasta_seqkit_stat_info': stat_info,
                    'fasta_type': seq_type,
                    'fasta_num_seqs': num_seqs
                }
                
            return {'error': 'File structure breaks seqkit parsing.'}
            
        except subprocess.CalledProcessError as e:
            return {'error': e.stderr}
        except FileNotFoundError:
            return {'error': 'Seqkit not found. Ensure it is installed and in PATH.'}

    def biopython_parser(self, seqkit_result) -> dict:
        if 'error' in seqkit_result:
            return seqkit_result

        seq_type = seqkit_result.get('fasta_type', '')
        
        if seq_type == 'Protein':
            db_name = 'uniprot'
            pattern = re.compile(r'[OPQ][0-9][A-Z0-9]{3}[0-9]|[A-NR-Z][0-9]([A-Z][A-Z0-9]{2}[0-9]){1,2}')
        elif seq_type in ['DNA', 'RNA']:
            db_name = 'ensembl'
            pattern = re.compile(r'ENS[A-Z]*[0-9]{11}|MGP_[A-Za-z0-9_]*_G[0-9]+')
        else:
            return {'error': f'Not supported sequence type: {seq_type}'}

        ultimate_output = {'DB_name': db_name}
        warnings = set()
        
        try:
            for s in SeqIO.parse(self.filename, 'fasta'):
                description = s.description
                match = pattern.search(description)
                
                if match:
                    extracted_id = match.group(0)
                    
                    # Fill file_info_{id}
                    ultimate_output[f'file_info_{extracted_id}'] = {
                        'description': description,
                        'sequence': str(s.seq)
                    }
                    
                    # Fetch and fill database_info_{id}
                    if db_name == 'uniprot':
                        resp = self._get_uniprot(extracted_id)
                        db_data = self._uniprot_parse_response(resp) if resp.status_code == 200 else {'error': 'API Error'}
                    else:
                        resp = self._get_ensembl(extracted_id)
                        db_data = self._ensembl_parse_response(resp) if resp.status_code == 200 else {'error': 'API Error'}
                    
                    # Remove redundant id generated by the _parse
                    inner_db_info = db_data.get(extracted_id, db_data)
                    ultimate_output[f'database_info_{extracted_id}'] = inner_db_info
                    
                else:
                    warnings.add('No ID match found.')
                    
            if warnings:
                ultimate_output['WARNING'] = warnings
                
        except Exception as e:
             return {'error': f'Failed reading FASTA file: {str(e)}'}

        return ultimate_output

    def show_output(self, output, indent=0):
        for key, value in output.items():
            print('\t' * indent + str(key))
            if isinstance(value, dict):
                self.show_output(value, indent + 1)
            else:
                print('\t' * (indent + 1) + str(value))

In [28]:
parser = MyFastaParser('test_file.fasta')
stats = parser.seqkit_stats()
# In the provided txt file json/object dumping is not called separately,
# but 'def seqkit_stats(self) -> dict' was hinting at just dictionary output,
# and I thought it indeed would be strange to always automatically dump
# (especially if someone would need to analyze a batch).
print(json.dumps(stats, indent=2))

{
  "fasta_seqkit_stat_info": {
    "file": "test_file.fasta",
    "format": "FASTA",
    "type": "Protein",
    "num_seqs": "2",
    "sum_len": "456",
    "min_len": "29",
    "avg_len": "228.0",
    "max_len": "427",
    "Q1": "29",
    "Q2": "228",
    "Q3": "427",
    "sum_gap": "0",
    "N50": "427",
    "N50_num": "1",
    "Q20(%)": "0",
    "Q30(%)": "0",
    "AvgQual": "0.00",
    "GC(%)": "0.00",
    "sum_n": "0"
  },
  "fasta_type": "Protein",
  "fasta_num_seqs": 2
}


In [29]:
biopython = parser.biopython_parser(stats)
parser.show_output(biopython)

DB_name
	uniprot
file_info_P11473
	description
		sp|P11473|VDR_HUMAN Vitamin D3 receptor OS=Homo sapiens OX=9606 GN=VDR PE=1 SV=1
	sequence
		MEAMAASTSLPDPGDFDRNVPRICGVCGDRATGFHFNAMTCEGCKGFFRRSMKRKALFTCPFNGDCRITKDNRRHCQACRLKRCVDIGMMKEFILTDEEVQRKREMILKRKEEEALKDSLRPKLSEEQQRIIAILLDAHHKTYDPTYSDFCQFRPPVRVNDGGGSHPSRPNSRHTPSFSGDSSSSCSDHCITSSDMMDSSSFSNLDLSEEDSDDPSVTLELSQLSMLPHLADLVSYSIQKVIGFAKMIPGFRDLTSEDQIVLLKSSAIEVIMLRSNESFTMDDMSWTCGNQDYKYRVSDVTKAGHSLELIEPLIKFQVGLKKLNLHEEEHVLLMAICIVSPDRPGVQDAALIEAIQDRLSNTLQTYIRCRHPPPGSHLLYAKMIQKLADLRSLNEEHSKQYRCLSFQPECSMKLTPLVLEVFGNEIS
database_info_P11473
	organism
		Homo sapiens
	geneInfo
		[{'geneName': {'evidences': [{'evidenceCode': 'ECO:0000312', 'source': 'HGNC', 'id': 'HGNC:12679'}], 'value': 'VDR'}, 'synonyms': [{'value': 'NR1I1'}]}]
	sequenceInfo
		value
			MEAMAASTSLPDPGDFDRNVPRICGVCGDRATGFHFNAMTCEGCKGFFRRSMKRKALFTCPFNGDCRITKDNRRHCQACRLKRCVDIGMMKEFILTDEEVQRKREMILKRKEEEALKDSLRPKLSEEQQRIIAILLDAHHKTYDPTYSDFCQFRPPVRVNDGGGSHPSRPNSRHTPSFSGDSSSSCSDHCITSS

In [30]:
parser_1 = MyFastaParser('ensembl_download_1.fasta')
stats_1 = parser_1.seqkit_stats()
print(json.dumps(stats_1, indent=2))

{
  "fasta_seqkit_stat_info": {
    "file": "ensembl_download_1.fasta",
    "format": "FASTA",
    "type": "DNA",
    "num_seqs": "6",
    "sum_len": "86",
    "min_len": "9",
    "avg_len": "14.3",
    "max_len": "23",
    "Q1": "10",
    "Q2": "14",
    "Q3": "17",
    "sum_gap": "0",
    "N50": "16",
    "N50_num": "3",
    "Q20(%)": "0",
    "Q30(%)": "0",
    "AvgQual": "0.00",
    "GC(%)": "45.35",
    "sum_n": "0"
  },
  "fasta_type": "DNA",
  "fasta_num_seqs": 6
}


In [31]:
biopython_1 = parser_1.biopython_parser(stats)
parser_1.show_output(biopython_1)

DB_name
	uniprot
file_info_D1OR15
	description
		ENST00000605284.1 cds chromosome:GRCh38:15:20011153:20011169:-1 gene:ENSG00000271336.1 gene_biotype:IG_D_gene transcript_biotype:IG_D_gene gene_symbol:IGHD1OR15-1A description:immunoglobulin heavy diversity 1/OR15-1A (non-functional) [Source:HGNC Symbol;Acc:HGNC:5487]
	sequence
		GGTATAACTGGAACAAC
database_info_D1OR15
	error
		API Error
file_info_D5OR15
	description
		ENST00000604642.1 cds chromosome:GRCh38:15:20003840:20003862:-1 gene:ENSG00000270961.1 gene_biotype:IG_D_gene transcript_biotype:IG_D_gene gene_symbol:IGHD5OR15-5A description:immunoglobulin heavy diversity 5/OR15-5A (non-functional) [Source:HGNC Symbol;Acc:HGNC:5512]
	sequence
		GTGGATATAGTGTCTACGATTAC
database_info_D5OR15
	error
		API Error
WARNING
	{'No ID match found.'}


In [32]:
parser_2 = MyFastaParser('ensembl_download_2.fasta')
stats_2 = parser_2.seqkit_stats()
print(json.dumps(stats_2, indent=2))

{
  "error": "File structure breaks seqkit parsing."
}


In [33]:
biopython_2 = parser_2.biopython_parser(stats)
parser_2.show_output(biopython_2)

DB_name
	uniprot
WARNING
	{'No ID match found.'}


c:\Users\Dastan\AppData\Local\Programs\Python\Python314\Lib\site-packages\Bio\SeqIO\FastaIO.py:202: BiopythonDeprecationWarning: Previously, the FASTA parser silently ignored comments at the beginning of the FASTA file (before the first sequence).

Nowadays, the FASTA file format is usually understood not to have any such comments, and most software packages do not allow them. Therefore, the use of comments at the beginning of a FASTA file is now deprecated in Biopython.

In a future Biopython release, this deprecation warning will be replaced by a ValueError. To avoid this, there are three options:

(1) Modify your FASTA file to remove such comments at the beginning of the file.

(2) Use SeqIO.parse with the 'fasta-pearson' format instead of 'fasta'. This format is consistent with the FASTA format defined by William Pearson's FASTA aligner software. This format allows for comments before the first sequence; lines starting with the ';' character anywhere in the file are also regarded a

In [34]:
parser_3 = MyFastaParser('uniprot_download.fasta')
stats_3 = parser_3.seqkit_stats()
print(json.dumps(stats_3, indent=2))

{
  "fasta_seqkit_stat_info": {
    "file": "uniprot_download.fasta",
    "format": "FASTA",
    "type": "Protein",
    "num_seqs": "7",
    "sum_len": "3861",
    "min_len": "180",
    "avg_len": "551.6",
    "max_len": "1382",
    "Q1": "429",
    "Q2": "441",
    "Q3": "500",
    "sum_gap": "0",
    "N50": "468",
    "N50_num": "3",
    "Q20(%)": "0",
    "Q30(%)": "0",
    "AvgQual": "0.00",
    "GC(%)": "0.00",
    "sum_n": "0"
  },
  "fasta_type": "Protein",
  "fasta_num_seqs": 7
}


In [35]:
biopython_3 = parser_3.biopython_parser(stats)
parser_3.show_output(biopython_3)

DB_name
	uniprot
file_info_Q9R1A7
	description
		sp|Q9R1A7|NR1I2_RAT Nuclear receptor subfamily 1 group I member 2 OS=Rattus norvegicus OX=10116 GN=Nr1i2 PE=2 SV=1
	sequence
		MRPEERWNHVGLVQREEADSVLEEPINVDEEDGGLQICRVCGDKANGYHFNVMTCEGCKGFFRRAMKRNVRLRCPFRKGTCEITRKTRRQCQACRLRKCLESGMKKEMIMSDAAVEQRRALIKRKKREKIEAPPPGGQGLTEEQQALIQELMDAQMQTFDTTFSHFKDFRLPAVFHSDCELPEVLQASLLEDPATWSQIMKDSVPMKISVQLRGEDGSIWNYQPPSKSDGKEIIPLLPHLADVSTYMFKGVINFAKVISHFRELPIEDQISLLKGATFEMCILRFNTMFDTETGTWECGRLAYCFEDPNGGFQKLLLDPLMKFHCMLKKLQLREEEYVLMQAISLFSPDRPGVVQRSVVDQLQERFALTLKAYIECSRPYPAHRFLFLKIMAVLTELRSINAQQTQQLLRIQDTHPFATPLMQELFSSTDG
database_info_Q9R1A7
	organism
		Rattus norvegicus
	geneInfo
		[{'geneName': {'value': 'Nr1i2'}, 'synonyms': [{'value': 'Pxr'}]}]
	sequenceInfo
		value
			MRPEERWNHVGLVQREEADSVLEEPINVDEEDGGLQICRVCGDKANGYHFNVMTCEGCKGFFRRAMKRNVRLRCPFRKGTCEITRKTRRQCQACRLRKCLESGMKKEMIMSDAAVEQRRALIKRKKREKIEAPPPGGQGLTEEQQALIQELMDAQMQTFDTTFSHFKDFRLPAVFHSDCELPEVLQASLLEDPATWSQIMKDSVPMKISVQLRGEDGSIWNYQPPSKSDGKEIIPLL